# M15 G4 — G-VIS (enmienda 1 de `docs/experiments/M15_G4_prior.md`)

Puerta previa a la generación completa. Entradas (solo lectura, solo las filas usadas):

- **G4**: la carpeta más reciente `processed/M15_G4/timing_*/signal/signal_shard00of01.h5`
  (salida de `m15_build_g4.py signal --limit 64`): `X_clean` (L(s_S15) en float16, antes del z-score),
  `row_index`, `y` ([chirp_mass, total_mass, chi_eff]), `edge_tapered`, `psd_key`, `status`.
- **G4**: `processed/M15_G4/proposal/train.npz` (`rows__source_index`, `rows__target_network_snr`, masas, χ_eff).
- **G3T (referencia descriptiva)**: `processed/M13_G3_band1024_w16_500k/m13_13_g3t/memmap/X_clean_train.npy`
  en las posiciones `default_rng(151106).choice(400000, 64)`.
- **Ruido (celda 5)**: `bank_X.npy` / `bank_psd_key.npy` de esa misma carpeta memmap (enlaces al banco de M13.6).

Definiciones (enmienda 1): E_r = Σ_d Σ_t L(s)²_{r,d,t} en float64 desde el float16 guardado; κ_r = E_r / ρ_r²,
con ρ_r el SNR objetivo de la fila.

In [ ]:
import csv
import json
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.paths import resolve_data_root

DATA_ROOT = resolve_data_root()
PROCESSED = DATA_ROOT / 'processed'
G4 = PROCESSED / 'M15_G4'
G3T_MM = PROCESSED / 'M13_G3_band1024_w16_500k' / 'm13_13_g3t' / 'memmap'
G1_H5 = PROCESSED / 'M11_final_G1_500k' / 'm11_final_G1_500k.h5'

DETECTORS = ('H1', 'L1', 'V1')        # scripts/generate_m11_final_G1.DETECTORS
LABELS = ('chirp_mass', 'total_mass', 'chi_eff')
FS = 4096.0                           # Hz, G1/G3 sampling_frequency
ZSCORE_EPS = 1e-6                     # train_m13_3.AUG_ZSCORE_EPS
G3T_SEED, G3T_N_TRAIN, N_REF = 151106, 400000, 64
M_BINS = [(0.0, 80.0, '< 80'), (80.0, 180.0, '80–180'), (180.0, np.inf, '180–250')]

# G4: most recent timing run with a signal shard (utc stamps sort lexicographically)
timing = sorted(p for p in G4.glob('timing_*') if (p / 'signal' / 'signal_shard00of01.h5').exists())
assert timing, f'No timing_*/signal/signal_shard00of01.h5 under {G4}'
SIG_H5 = timing[-1] / 'signal' / 'signal_shard00of01.h5'
with h5py.File(SIG_H5, 'r') as h:
    status = h['status'][:]
    assert (status == 1).all(), f'{SIG_H5}: {(status != 1).sum()} rows not written'
    X4 = h['X_clean'][:]                       # (64, 3, T) float16
    rid4 = h['row_index'][:].astype(np.int64)
    y4 = h['y'][:].astype(np.float64)
    edge4 = h['edge_tapered'][:].astype(bool)
    key4 = h['psd_key'].asstr()[:]
T = X4.shape[-1]
print(f'G4 signal: {SIG_H5}  {X4.shape} {X4.dtype}')

with np.load(G4 / 'proposal' / 'train.npz') as z:
    sid = z['rows__source_index'].astype(np.int64)
    assert (np.diff(sid) > 0).all()
    j = np.searchsorted(sid, rid4)
    assert np.array_equal(sid[np.minimum(j, sid.size - 1)], rid4), 'row_index not in proposal/train.npz'
    rho4 = z['rows__target_network_snr'][j].astype(np.float64)
    m1_4, m2_4 = z['rows__mass_1'][j].astype(np.float64), z['rows__mass_2'][j].astype(np.float64)
    mc4, mt4, chi4 = (z[f'rows__{k}'][j].astype(np.float64) for k in LABELS)
assert np.allclose(np.c_[mc4, mt4, chi4], y4, rtol=1e-6, atol=1e-6), 'y of the shard differs from train.npz'
q4 = m2_4 / m1_4

### Procedencia del SNR objetivo de G3T

- Fila de G3T en la posición p: `row_ids_train.npy[p]` de la carpeta `g3_memmap.dir` que registra
  `m13_13_g3t/memmap/manifest.json` (el mismo orden que `X_clean_train`, comprobado en el merge de m13_13).
- ρ = `target_network_snr` del manifiesto G1 realmente usado (ruta en `attrs['manifest_path']` del HDF5 G1),
  en la fila con `source_index` igual a ese id (en G1 `source_index == manifest_index == fila`).
- **Banda**: G3 reconstruye las filas G1 con la física G3 (`configs/generation/generate_m13_G3_band1024_w16.json`,
  override `simulation.snr_high_frequency_cutoff = 1024`), así que `build_g1_row` reescaló la distancia a ese ρ en
  **30–1024 Hz** (el manifiesto G1 original lo definía en 30–512 Hz; el valor numérico es el mismo, la banda no).
  G4 usa la misma física G3, así que su ρ también es 30–1024 Hz. En ambos L(s) es la señal S15 (desde 15 Hz), por lo
  que κ incluye la parte 15–30 Hz igual en G3T y G4.

In [ ]:
pos3 = np.random.default_rng(G3T_SEED).choice(G3T_N_TRAIN, N_REF)   # literal; with replacement (default)
print(f'G3T positions: {pos3.size} ({np.unique(pos3).size} unique)')
X3_mm = np.load(G3T_MM / 'X_clean_train.npy', mmap_mode='r')
assert X3_mm.shape[0] == G3T_N_TRAIN, X3_mm.shape
order = np.argsort(pos3, kind='stable')                            # monotone memmap reads
X3 = np.empty((pos3.size, *X3_mm.shape[1:]), dtype=X3_mm.dtype)
X3[order] = X3_mm[pos3[order]]
g3_mm_dir = Path(json.loads((G3T_MM / 'manifest.json').read_text())['g3_memmap']['dir'])
rid3 = np.load(g3_mm_dir / 'row_ids_train.npy', mmap_mode='r')[pos3].astype(np.int64)

with h5py.File(G1_H5, 'r') as h:
    g1_manifest = Path(h.attrs['manifest_path'])
want = {int(r) for r in rid3}
found = {}
with g1_manifest.open(newline='') as fh:                           # streamed: only the 64 rows are kept
    for rec in csv.DictReader(fh):
        si = int(rec['source_index'])
        if si in want:
            found[si] = (float(rec['target_network_snr']), float(rec['total_mass']))
assert len(found) == len(want), 'G3T rows missing from the G1 manifest'
rho3 = np.array([found[int(r)][0] for r in rid3])
mt3 = np.array([found[int(r)][1] for r in rid3])
print(f'G1 manifest: {g1_manifest}')

## (1) Tabla por fila (G4)

In [ ]:
def energy(X):
    # Sum over detectors and samples, float64 from the stored float16 (before the z-score).
    x = np.asarray(X, dtype=np.float64)
    return (x * x).sum(axis=(1, 2))

E4, E3 = energy(X4), energy(X3)
k4, k3 = E4 / rho4**2, E3 / rho3**2

hdr = f"{'row_index':>10} {'M_tot':>7} {'Mc':>7} {'q':>6} {'chi_eff':>8} {'rho':>6} {'E':>10} {'kappa':>8} edge"
print(hdr)
print('-' * len(hdr))
for i in np.argsort(mt4, kind='stable'):
    print(f'{rid4[i]:>10d} {mt4[i]:7.2f} {mc4[i]:7.2f} {q4[i]:6.3f} {chi4[i]:+8.3f} {rho4[i]:6.2f} '
          f'{E4[i]:10.2f} {k4[i]:8.4f} {int(edge4[i]):>4d}')

## (2) Veredicto G-VIS

(i) la mediana de κ por bin de M_tot (< 80, 80–180, 180–250) está dentro de ±10 % de la mediana global;
(ii) ninguna fila tiene κ < 0,5 × mediana global. Mediana global sobre las 64 filas G4. Un bin sin filas se marca
como no evaluable y el veredicto queda INDETERMINADO (no PASA).

In [ ]:
med = float(np.median(k4))
print(f'G4 rows: {k4.size}; global median kappa = {med:.5f}')
ok_i, evaluable = True, True
for lo, hi, name in M_BINS:
    sel = (mt4 >= lo) & (mt4 < hi)
    if not sel.any():
        evaluable = False
        print(f'  (i) M_tot {name:>8}: n = 0  -> not evaluable')
        continue
    mb = float(np.median(k4[sel]))
    rel = mb / med - 1.0
    ok = abs(rel) <= 0.10
    ok_i &= ok
    print(f'  (i) M_tot {name:>8}: n = {int(sel.sum()):2d}, median kappa = {mb:.5f}, '
          f'rel. to global = {100 * rel:+6.2f} %  (|.| <= 10 %: {ok})')
low = k4 < 0.5 * med
ok_ii = not low.any()
print(f'  (ii) min kappa = {k4.min():.5f} = {k4.min() / med:.3f} x median; rows below 0.5 x median: '
      f'{int(low.sum())} {list(rid4[low])}  -> {ok_ii}')
verdict = 'INDETERMINADO' if not evaluable else ('PASA' if (ok_i and ok_ii) else 'FALLA')
print(f'G-VIS: {verdict}')
print(f'(descriptivo) G3T: median kappa = {np.median(k3):.5f} on {k3.size} rows; '
      f'G4/G3T median ratio = {med / np.median(k3):.4f}')

## (3) κ frente a M_tot (G4 y G3T, descriptivo)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(mt3, k3, s=18, marker='s', alpha=0.6, label=f'G3T (n={k3.size})')
ax.scatter(mt4[~edge4], k4[~edge4], s=22, label=f'G4 (n={int((~edge4).sum())})')
if edge4.any():
    ax.scatter(mt4[edge4], k4[edge4], s=40, facecolors='none', edgecolors='k', label='G4 edge_tapered')
ax.axhline(med, color='k', lw=0.8, label='G4 global median')
ax.axhspan(0.9 * med, 1.1 * med, color='k', alpha=0.08, label='±10 %')
ax.axhline(0.5 * med, color='r', lw=0.8, ls='--', label='0.5 × median')
for edge in (80, 180):
    ax.axvline(edge, color='0.6', lw=0.6, ls=':')
ax.set_xlabel(r'$M_{\rm tot}$ [M$_\odot$]')
ax.set_ylabel(r'$\kappa = E/\rho^2$')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## (4) L(s) por detector: M_tot mínima, la más cercana a 150 y la máxima

In [ ]:
picks = [int(np.argmin(mt4)), int(np.argmin(np.abs(mt4 - 150.0))), int(np.argmax(mt4))]
t = np.arange(T) / FS
HALF = 0.25   # s: 0.5 s zoom window

def zoom_slice(x):
    c = int(np.argmax((np.asarray(x, dtype=np.float64) ** 2).sum(axis=0)))  # peak of sum_d L^2
    a, b = max(0, c - int(HALF * FS)), min(T, c + int(HALF * FS))
    return slice(a, b)

fig, axes = plt.subplots(len(picks), 2, figsize=(12, 2.6 * len(picks)), squeeze=False)
for r, i in enumerate(picks):
    x = X4[i].astype(np.float64)
    sl = zoom_slice(x)
    for d, ifo in enumerate(DETECTORS):
        axes[r, 0].plot(t, x[d], lw=0.6, label=ifo)
        axes[r, 1].plot(t[sl], x[d, sl], lw=0.8, label=ifo)
    axes[r, 0].set_title(f'row {rid4[i]}: M_tot={mt4[i]:.1f}, q={q4[i]:.2f}, chi_eff={chi4[i]:+.2f}, '
                         f'rho={rho4[i]:.1f}, edge={int(edge4[i])}', fontsize=9)
    axes[r, 1].set_title('zoom 0.5 s around the maximum', fontsize=9)
    axes[r, 0].set_ylabel('L(s)')
    axes[r, 0].legend(fontsize=7, loc='upper left')
for ax in axes[-1]:
    ax.set_xlabel('t [s]')
plt.tight_layout()
plt.show()

## (5) Las mismas 3 filas con ruido del banco

x = L(s) + n, con n = `bank_X[j·K]` (primera realización de su `psd_key`; bloques de K entradas consecutivas por
`psd_key` ordenado, como `train_m13_3.load_noise_aug`), signo +1 y sin inversión temporal; suma en float32 y z-score
por muestra y detector (media, desviación poblacional + ε = 1e-6), como `train_m13_3.augment_on_gpu`, en numpy.

In [ ]:
bank_keys = np.load(G3T_MM / 'bank_psd_key.npy').astype(str)
K = int(np.sum(bank_keys == bank_keys[0]))
kb = bank_keys.reshape(-1, K)
uk = kb[:, 0]
assert (kb == uk[:, None]).all() and (uk[1:] > uk[:-1]).all(), 'bank_psd_key is not K consecutive per sorted key'
bank_X = np.load(G3T_MM / 'bank_X.npy', mmap_mode='r')

fig, axes = plt.subplots(len(picks), 2, figsize=(12, 2.6 * len(picks)), squeeze=False)
for r, i in enumerate(picks):
    jk = int(np.searchsorted(uk, key4[i]))
    assert jk < uk.size and uk[jk] == key4[i], f'psd_key {key4[i]} not in the bank'
    noise = np.asarray(bank_X[jk * K], dtype=np.float32)          # first realization of the key
    x = X4[i].astype(np.float32) + noise
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)                            # population (ddof=0)
    xz = (x - mean) / (std + np.float32(ZSCORE_EPS))
    sl = zoom_slice(X4[i])                                         # same window as (4)
    for d, ifo in enumerate(DETECTORS):
        axes[r, 0].plot(t, xz[d], lw=0.4, label=ifo)
        axes[r, 1].plot(t[sl], xz[d, sl], lw=0.7, label=ifo)
    axes[r, 0].set_title(f'row {rid4[i]}: M_tot={mt4[i]:.1f}, psd_key={key4[i]}, bank entry {jk * K}', fontsize=9)
    axes[r, 1].set_title('zoom 0.5 s (window of the clean maximum)', fontsize=9)
    axes[r, 0].set_ylabel('z-scored X')
    axes[r, 0].legend(fontsize=7, loc='upper left')
for ax in axes[-1]:
    ax.set_xlabel('t [s]')
plt.tight_layout()
plt.show()